# 02 Analysis

This notebook uses the processed data and summary CSV files to reproduce the core analysis behind the interactive report in `docs/index.html`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "code":
    ROOT_DIR = ROOT_DIR.parent

CODE_DIR = ROOT_DIR / "code"
DATA_DIR = ROOT_DIR / "data"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from project_utils import build_geography_map_figure

processed = pd.read_csv(DATA_DIR / "acled_mena_processed.csv", parse_dates=["event_date"])
yearly = pd.read_csv(DATA_DIR / "yearly_shift_summary.csv")
spatial = pd.read_csv(DATA_DIR / "spatial_bucket_summary.csv")
country = pd.read_csv(DATA_DIR / "country_pattern_summary.csv")

report_df = processed[processed["year"].between(1997, 2024)].copy()
report_df.shape


## Research question

Has the rise of drones and other remote attacks in the Middle East and North Africa been accompanied by a spatial shift from border-proximate conflict toward attacks on capital areas?

The analysis is descriptive rather than causal: it evaluates whether the observed spatial distribution of events is consistent with a shift toward capital targeting.


In [ ]:
headline = {
    "all_events": len(report_df),
    "border_proximate_events": int(report_df["is_border_proximate"].sum()),
    "border_proximate_share_pct": round(report_df["is_border_proximate"].mean() * 100, 2),
    "capital_area_events": int(report_df["is_capital_area"].sum()),
    "capital_area_share_pct": round(report_df["is_capital_area"].mean() * 100, 2),
    "capital_remote_events": int(report_df["capital_remote_event"].sum()),
    "capital_remote_share_of_remote_pct": round(
        report_df["capital_remote_event"].sum() / report_df["is_remote"].sum() * 100, 2
    ),
}

headline


## Figure 1: Within remote attacks

Border-proximate remote violence remains much more common than capital-area remote violence across most of the period.


In [ ]:
remote_yearly = (
    report_df.groupby("year")
    .agg(
        all_remote_events=("is_remote", "sum"),
        border_remote_events=("is_border_proximate", lambda x: int((x & report_df.loc[x.index, "is_remote"]).sum())),
        capital_remote_events=("capital_remote_event", "sum"),
    )
    .reset_index()
)
remote_yearly["border_remote_share_pct"] = (
    remote_yearly["border_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)
remote_yearly["capital_remote_share_pct"] = (
    remote_yearly["capital_remote_events"] / remote_yearly["all_remote_events"] * 100
).round(2)

fig1 = go.Figure()
fig1.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["border_remote_share_pct"],
        mode="lines+markers",
        name="Border-proximate share of remote attacks",
        line={"color": "#1f4e79", "width": 3.2},
    )
)
fig1.add_trace(
    go.Scatter(
        x=remote_yearly["year"],
        y=remote_yearly["capital_remote_share_pct"],
        mode="lines+markers",
        name="Capital-area share of remote attacks",
        line={"color": "#c44536", "width": 3.0},
    )
)
fig1.update_layout(
    title="Within remote attacks, border-proximate share remains above capital-area share",
    template="plotly_white",
    xaxis_title="Year",
    yaxis_title="Share (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
fig1.show()


## Figure 2: Yearly all-event denominator

This figure uses yearly shares of all events. The blue segment uses non-capital border-proximate events so the stacked bars do not double-count locations that are both capital-area and border-proximate.


In [ ]:
yearly_geo = (
    report_df.assign(noncapital_border=report_df["is_border_proximate"] & ~report_df["is_capital_area"])
    .groupby("year")
    .agg(
        all_events=("event_id_cnty", "count"),
        noncapital_border_events=("noncapital_border", "sum"),
        capital_area_events=("is_capital_area", "sum"),
    )
    .reset_index()
)
yearly_geo["noncapital_border_share_pct"] = (
    yearly_geo["noncapital_border_events"] / yearly_geo["all_events"] * 100
).round(2)
yearly_geo["capital_area_share_pct"] = (
    yearly_geo["capital_area_events"] / yearly_geo["all_events"] * 100
).round(2)

fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["noncapital_border_share_pct"],
        name="Non-capital border-proximate share",
        marker={"color": "#1f4e79"},
    )
)
fig2.add_trace(
    go.Bar(
        x=yearly_geo["year"],
        y=yearly_geo["capital_area_share_pct"],
        name="Capital-area share",
        marker={"color": "#c44536"},
    )
)
fig2.update_layout(
    title="Yearly all-event shares: border-proximate space remains heavier than capital space",
    template="plotly_white",
    barmode="stack",
    xaxis_title="Year",
    yaxis_title="Share of all events (%)",
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
fig2.show()


## Summary tables

The summary CSV files provide the aggregate values used in the report narrative.


In [ ]:
spatial


In [ ]:
country.head(10)


## Figure 4 map

The published HTML report uses a simplified yearly map slider. The helper below builds the full Plotly map from the processed event-level data.


In [ ]:
# This can be browser-heavy because it plots event-level geography.
# Uncomment to render inside the notebook.
# build_geography_map_figure(report_df).show()


## Conclusion

The evidence supports one stable conclusion: border-linked space remains the heavier conflict geography in MENA relative to capital space. The data do not support a simple regional story in which conflict moved from borders to capitals.
